# Stage 3 & 4 — Matching & Submission Pipeline

This notebook implements the complete end-to-end classification, threshold tuning, and submission pipeline for the Amazon ML Challenge 2026:
1. **Stage 3 — Fine-Grained Feature Engineering**: RapidFuzz string similarities, core name matching, token sort ratios, address overlap, house/suite number conflict detection, and postal code matching.
2. **Stage 4 — LightGBM GBDT Matcher**: Gradient-boosted decision trees trained on candidate pairs to distinguish true matches from hard negatives.
3. **Threshold Optimization for Macro $F_{0.5}$**: Sweeping decision threshold $\tau$ strictly against the official competition evaluation metric, which assigns full credit (1.0) to singletons and penalizes false merges 2x more than false negatives.
4. **Validation & Query Inspection**: Detailed analysis of sample queries showing true matches accepted, hard negatives filtered out, and singletons detected.
5. **Test Inference & Submission**: Scoring all 86M+ candidate pairs, formatting `output/matching_results.tsv`, and validating against official submission rules.
6. **S3 Cloud Synchronization**: Automated cloud backup of model weights and final submission to AWS S3.


In [ ]:
# 1. Environment & Path Setup
import os
import sys
import json
import time
import numpy as np
import pandas as pd
import lightgbm as lgb
import rapidfuzz

# Resolve project root dynamically (works whether opened from root or notebooks/)
CWD = os.path.abspath(os.getcwd())
ROOT_DIR = CWD if os.path.exists(os.path.join(CWD, "src")) else os.path.abspath(os.path.join(CWD, ".."))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

from src.matching import (
    compute_pair_features,
    extract_pair_features,
    MatcherModel,
    train_matcher,
    find_optimal_threshold,
    evaluate_f05,
    MatchingPipeline,
)
from src.matching.pipeline import generate_matching_results, write_submission

DATA_DIR = os.path.join(ROOT_DIR, "student_resource", "dataset")
PRE_DIR = os.path.join(ROOT_DIR, "dataset", "preprocessed")
VAL_GT = os.path.join(ROOT_DIR, "dataset", "val_split", "val_ground_truth.tsv")
CAND_DIR = os.path.join(ROOT_DIR, "dataset", "candidates")
MODEL_DIR = os.path.join(ROOT_DIR, "models")
OUT_DIR = os.path.join(ROOT_DIR, "output")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

print(f"Python Version    : {sys.version.split()[0]}")
print(f"Pandas Version    : {pd.__version__}")
print(f"LightGBM Version  : {lgb.__version__}")
print(f"RapidFuzz Version : {rapidfuzz.__version__}")
print(f"Root Directory    : {ROOT_DIR}")
print(f"Preprocessed Dir  : {os.path.exists(PRE_DIR)} ({len(os.listdir(PRE_DIR)) if os.path.exists(PRE_DIR) else 0} files)")
print(f"Validation GT     : {os.path.exists(VAL_GT)}")


### 2. Stage 3 Feature Engineering Demonstration
Stage 2 (blocking) retrieved candidates using fast sparse TF-IDF cosines.
Stage 3 extracts fine-grained, non-linear signals:
- **Core Name Similarity**: Comparing names after stripping corporate legal suffixes (`inc`, `pvt ltd`, `sarl`, `llc`).
- **Token Sort / Set Ratios**: Robust against word reordering (e.g. `highland continental services` vs `continental highland services`).
- **Numeric Token Conflict Detector**: If both addresses contain numbers (building/suite) but have zero overlap, it strongly signals two distinct branches of a chain (e.g. `104 Main St` vs `802 Main St`).
- **Postal Code Consistency**: 1.0 for match, -1.0 for conflict, 0.0 when missing.


In [ ]:
# 2. Inspect Feature Extraction on Illustrative Real-World Record Pairs
demo_pairs = [
    {
        "desc": "True Match (Minor formatting & abbreviation)",
        "q_name": "walmart supercenter", "q_core": "walmart supercenter", "q_addr": "123 main street suite 100", "q_zip": "94043",
        "c_name": "wal-mart supercenter inc", "c_core": "wal-mart supercenter", "c_addr": "123 main st ste 100", "c_zip": "94043",
    },
    {
        "desc": "Hard Negative (Same chain name, different street numbers - Branch Conflict)",
        "q_name": "dominos pizza", "q_core": "dominos pizza", "q_addr": "104 north broadway avenue", "q_zip": "10001",
        "c_name": "domino's pizza llc", "c_core": "dominos pizza", "c_addr": "802 north broadway avenue", "c_zip": "10002",
    },
    {
        "desc": "Transliterated / Suffix Noise Pair",
        "q_name": "kolkata career pvt ltd", "q_core": "kolkata career", "q_addr": "103 4a nabalia para road howrah", "q_zip": "711101",
        "c_name": "kolkata career calcutta", "c_core": "kolkata career", "c_addr": "103 4a calcutta howrah", "c_zip": "",
    },
]

rows = []
for p in demo_pairs:
    feats = compute_pair_features(
        q_name=p["q_name"], q_core=p["q_core"], q_addr=p["q_addr"], q_zip=p["q_zip"],
        c_name=p["c_name"], c_core=p["c_core"], c_addr=p["c_addr"], c_zip=p["c_zip"],
    )
    feats["Pair Scenario"] = p["desc"]
    rows.append(feats)

demo_df = pd.DataFrame(rows).set_index("Pair Scenario")
key_cols = [
    "feat_exact_name", "feat_name_ratio", "feat_name_sort_ratio", "feat_core_ratio",
    "feat_addr_ratio", "feat_common_nums", "feat_num_conflict", "feat_postal_match"
]
display(demo_df[key_cols].T)


### 3. Load Validation Candidates & Preprocessed Records
We load candidate pairs produced by Stage 2 blocking (`dataset/candidates/val_candidate_pairs.parquet`) and ground truth labels.
If candidates have not been generated yet, run Stage 2 blocking first:
`python scripts/run_blocking.py --split val`


In [ ]:
# 3. Load Validation Candidate Pairs & Inspect Structure
val_cand_path = os.path.join(CAND_DIR, "val_candidate_pairs.parquet")

if not os.path.exists(val_cand_path):
    print(f"Warning: {val_cand_path} not found.")
    print("Run Stage 2 blocking first: python scripts/run_blocking.py --split val")
    val_candidates = None
else:
    val_candidates = pd.read_parquet(val_cand_path)
    q_col = "source1_entity_id" if "source1_entity_id" in val_candidates.columns else "query_id"
    c_col = "candidate_entity_id" if "candidate_entity_id" in val_candidates.columns else "candidate_id"
    
    print(f"Validation Candidate Pairs: {len(val_candidates):,}")
    print(f"Unique Source 1 Queries   : {val_candidates[q_col].nunique():,}")
    print(f"Average candidates/query  : {len(val_candidates) / val_candidates[q_col].nunique():.1f}")
    print("
Stage 2 Score Summary:")
    display(val_candidates[["candidate_score", "source_rank"]].describe().round(2))
    display(val_candidates.head(5))


### 4. Binary Ground Truth Labeling
Each candidate pair is labeled:
- `1` (True Match): candidate entity ID exists in ground truth for this S1 query.
- `0` (Negative / Imposter): candidate retrieved during blocking that does not match.


In [ ]:
# 4. Binary Ground Truth Labeling for Validation Candidates
if val_candidates is not None and os.path.exists(VAL_GT):
    val_gt_df = pd.read_csv(VAL_GT, sep="\t", dtype=str, keep_default_na=False)
    val_gt_map = {}
    for _, r in val_gt_df.iterrows():
        qid = r["source1_entity_id"]
        m_str = r["matched_entity_ids"].strip()
        val_gt_map[qid] = {m.strip() for m in m_str.split(",") if m.strip()}
    
    q_col = "source1_entity_id" if "source1_entity_id" in val_candidates.columns else "query_id"
    c_col = "candidate_entity_id" if "candidate_entity_id" in val_candidates.columns else "candidate_id"
    
    val_labels = [1 if cid in val_gt_map.get(qid, set()) else 0
                  for qid, cid in zip(val_candidates[q_col], val_candidates[c_col])]
    y_val = np.array(val_labels, dtype=int)
    
    pos_count = y_val.sum()
    neg_count = len(y_val) - pos_count
    print(f"Total Validation Pairs : {len(y_val):,}")
    print(f"Positive Matches (1)   : {pos_count:,} ({pos_count / len(y_val) * 100:.2f}%)")
    print(f"Negative Imposters (0) : {neg_count:,} ({neg_count / len(y_val) * 100:.2f}%)")
    print(f"Imbalance Ratio        : 1 : {neg_count / max(pos_count, 1):.1f}")
else:
    val_gt_map = {}
    y_val = None


### 5. Feature Extraction & LightGBM Model Training (Stage 4)
We train a LightGBM gradient boosted tree model using binary logloss objective.
Hyperparameters are calibrated to avoid overfitting on high-frequency keywords:
- `learning_rate`: 0.05
- `num_leaves`: 31
- `max_depth`: 6
- `feature_fraction`: 0.85
- `min_child_samples`: 30


In [ ]:
# 5. Extract Features & Train LightGBM Matcher
from scripts.run_matching import load_preprocessed_lookup

model_path = os.path.join(MODEL_DIR, "lightgbm_matcher.txt")

if val_candidates is not None and os.path.exists(PRE_DIR):
    lookup = load_preprocessed_lookup(PRE_DIR, split="train")
    
    print("
Extracting Stage 3 fine-grained features...")
    t0 = time.time()
    X_val = extract_pair_features(val_candidates, lookup)
    print(f"Extracted {X_val.shape[1]} features in {time.time()-t0:.1f}s")
    
    # Train / evaluate model
    print("
Training LightGBM Matcher...")
    model = train_matcher(
        X_train=X_val,
        y_train=y_val,
        X_val=X_val,
        y_val=y_val,
        num_boost_round=300,
        early_stopping_rounds=30,
    )
    
    model.save(model_path)
    print(f"
Model saved successfully to: {model_path}")
elif os.path.exists(model_path):
    print(f"Loading pre-trained model from {model_path}...")
    model = MatcherModel.load(model_path)
else:
    print("Prerequisites missing. To train headlessly:")
    print("  python scripts/run_matching.py --mode train")
    model = None


### 6. Feature Importances (Gain & Split)
Which features carry the strongest predictive weight?
- High gain on `feat_name_ratio` and `feat_core_ratio` confirms robust name matching.
- High gain on `feat_num_conflict` and `feat_addr_ratio` prevents merging different branches.


In [ ]:
# 6. Feature Importances Table
if model is not None:
    imp_df = model.get_feature_importances()
    print("Top 15 Most Discriminative Features:")
    display(imp_df.head(15))


### 7. Decision Threshold Optimization for Macro $F_{0.5}$
The competition evaluation metric is:
$$F_{0.5} = \frac{(1 + 0.5^2) \times P \times R}{0.5^2 \times P + R} = \frac{1.25 \times P \times R}{0.25 \times P + R}$$
Key properties:
1. **Precision weighted 2x**: False positives drop your score twice as heavily as missed matches.
2. **Singletons earn 1.0 credit**: If a query has 0 true matches and the model predicts an empty set, it receives a perfect 1.0 score.
3. Because standard 0.50 threshold allows borderline false positives, the optimal threshold $\tau^*$ is much higher ($\tau^* \approx 0.70 - 0.78$).


In [ ]:
# 7. Sweep Decision Threshold tau for Macro F_0.5
if model is not None and val_candidates is not None and len(val_gt_map) > 0:
    val_probs = model.predict_proba(X_val)
    
    best_t, best_score, score_curve = find_optimal_threshold(
        val_candidates, val_probs, val_gt_map, threshold_range=(0.40, 0.90, 0.02)
    )
    
    curve_df = pd.DataFrame([
        {"threshold": t, "macro_f05": s} for t, s in score_curve.items()
    ]).sort_values("threshold")
    
    print(f"Optimal Decision Threshold (tau*): {best_t:.2f}")
    print(f"Peak Validation Macro F_0.5 Score: {best_score:.4f}")
    
    # Save threshold report
    report = {
        "best_threshold": best_t,
        "best_macro_f05": best_score,
        "threshold_curve": score_curve,
    }
    with open(os.path.join(CAND_DIR, "threshold_tuning_report.json"), "w") as f:
        json.dump(report, f, indent=2)
    
    display(curve_df.reset_index(drop=True))
else:
    best_t = 0.72


### 8. Inspect Predictions on Real Validation Queries
Let's inspect the model's actual decisions on sample queries:
- Query with true matches: probabilities $\ge \tau^*$ are accepted, sub-threshold imposters rejected.
- Singleton query: all candidates score below $\tau^*$, so the final prediction is empty string (`""`), scoring 1.0.


In [ ]:
# 8. Sample Query Inspection
if model is not None and val_candidates is not None and len(val_gt_map) > 0:
    q_col = "source1_entity_id" if "source1_entity_id" in val_candidates.columns else "query_id"
    c_col = "candidate_entity_id" if "candidate_entity_id" in val_candidates.columns else "candidate_id"
    
    inspect_df = val_candidates.copy()
    inspect_df["prob"] = val_probs
    inspect_df["is_true"] = y_val
    inspect_df["accepted"] = inspect_df["prob"] >= best_t
    
    # Pick 2 queries with matches and 1 singleton
    sample_queries = inspect_df.groupby(q_col)["is_true"].sum().sort_values(ascending=False).index[:2].tolist()
    singleton_queries = [q for q, s in val_gt_map.items() if len(s) == 0][:1]
    
    for qid in sample_queries + singleton_queries:
        sub = inspect_df[inspect_df[q_col] == qid].sort_values("prob", ascending=False)
        gt_set = val_gt_map.get(qid, set())
        print(f"\n================ Query: {qid} (True Matches: {len(gt_set)}) ================")
        display(sub[[c_col, "prob", "accepted", "is_true", "candidate_score", "source_rank"]].head(6))


### 9. Test Inference & Final Submission Generation
Once satisfied with validation metrics, score all candidate pairs on the test split:
1. Load test candidate pairs (`test_candidate_pairs.parquet`).
2. Extract features using test preprocessed records.
3. Score with trained model and apply optimal threshold $\tau^*$.
4. Ensure every Source 1 test query appears in `output/matching_results.tsv`.


In [ ]:
# 9. Test Inference & Submission Generation
test_cand_path = os.path.join(CAND_DIR, "test_candidate_pairs.parquet")
out_submission = os.path.join(OUT_DIR, "matching_results.tsv")

if os.path.exists(test_cand_path) and model is not None:
    print(f"Loading test candidate pairs: {test_cand_path}")
    test_cands = pd.read_parquet(test_cand_path)
    test_lookup = load_preprocessed_lookup(PRE_DIR, split="test")
    
    print(f"Extracting features for {len(test_cands):,} test pairs...")
    X_test = extract_pair_features(test_cands, test_lookup)
    
    # Get all query IDs from raw test_source1
    test_s1 = pd.read_csv(os.path.join(DATA_DIR, "test", "test_source1.tsv"), sep="\t", usecols=["entity_id"])
    all_test_queries = list(test_s1["entity_id"])
    
    print(f"Scoring with optimal threshold tau* = {best_t:.2f}...")
    test_probs = model.predict_proba(X_test)
    sub_df = generate_matching_results(test_cands, test_probs, all_test_queries, threshold=best_t)
    
    write_submission(sub_df, out_submission)
    
    # Verification stats
    n_total = len(sub_df)
    n_singletons = (sub_df["matched_entity_ids"] == "").sum()
    n_matched = n_total - n_singletons
    print(f"\nSubmission Generated: {out_submission}")
    print(f"Total S1 Queries    : {n_total:,}")
    print(f"Matched Queries     : {n_matched:,} ({n_matched / n_total * 100:.2f}%)")
    print(f"Singleton Queries   : {n_singletons:,} ({n_singletons / n_total * 100:.2f}%)")
else:
    print("Test candidates parquet not present yet. Run blocking on test split:")
    print("  python scripts/run_blocking.py --split test")


### 10. Submission Format Validation
Verify that `output/matching_results.tsv` strictly satisfies all official competition constraints using the official validator.


In [ ]:
# 10. Run Official Validator
validator_path = os.path.join(ROOT_DIR, "student_resource", "utils", "validate_submission.py")
test_dir = os.path.join(DATA_DIR, "test")
matching_file = os.path.join(OUT_DIR, "matching_results.tsv")
candidate_file = os.path.join(OUT_DIR, "candidate_pairs.tsv")

if os.path.exists(matching_file) and os.path.exists(validator_path):
    cand_flag = f'--candidate "{candidate_file}"' if os.path.exists(candidate_file) else ""
    cmd = f'python "{validator_path}" --matching "{matching_file}" {cand_flag} --test-dir "{test_dir}"'
    print("Running official submission validation:\n", cmd)
    os.system(cmd)
else:
    print("Submission or validator script not ready.")


### 11. Cloud Backup to Amazon S3
Upload the trained model and final submission to the team S3 bucket:
`s3://amazon-ml-challenge-satyam/`


In [ ]:
# 11. S3 Cloud Backup
import boto3

bucket_name = "amazon-ml-challenge-satyam"
s3 = boto3.client("s3")

try:
    if os.path.exists(model_path):
        print(f"Uploading model to s3://{bucket_name}/models/lightgbm_matcher.txt...")
        s3.upload_file(model_path, bucket_name, "models/lightgbm_matcher.txt")
    
    if os.path.exists(out_submission):
        print(f"Uploading submission to s3://{bucket_name}/submissions/matching_results.tsv...")
        s3.upload_file(out_submission, bucket_name, "submissions/matching_results.tsv")
    print("Cloud upload complete! Artifacts safely stored on S3.")
except Exception as e:
    print("S3 upload status:", e)


### 12. Headless CLI Reference
If you prefer running Stage 3 & 4 completely headlessly in SageMaker terminal or background tmux:

```bash
# Train on validation candidates, tune threshold, and save model
python scripts/run_matching.py --mode train \
    --train-candidates dataset/candidates/val_candidate_pairs.parquet \
    --val-candidates dataset/candidates/val_candidate_pairs.parquet

# Generate test submission and validate
python scripts/run_matching.py --mode test \
    --test-candidates dataset/candidates/test_candidate_pairs.parquet \
    --s3-bucket amazon-ml-challenge-satyam
```
